# Revision temas 3-4

# 🔤 Introducción

## 🧰 Carga de todas las librerías necesarias

In [1]:
# cargando todas las librerías necesarias
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from tabulate import tabulate
# scales
from sklearn.preprocessing import MinMaxScaler
# labels categoricos
from sklearn.preprocessing import LabelEncoder

# pca
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

# series temporales
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error

# warnings
import warnings
warnings.filterwarnings('ignore')

print("Librerias importadas!")

Librerias importadas!


In [2]:
# estilo de las graficas
sns.set_style("white")
# colores tecmilenio
_color_tecmi_light="#26d07c"
_color_tecmi_dark="#00534c"
_color_teal="#20c997"
# colores generales
_color_purple="#6f42c1"
_color_pink="#e83e8c"
_color_orange="#fd7e14"
# colores boostrap
_color_primary="#007bff"
_color_success="#28a745"
_color_info="#17a2b8"
_color_warning="#ffc107"
# tamaños de fuente
_fontsize_title = 16
_fontsize_label = 12
_fontsize_marker = 9
# paletas de colores prearmadas
_paleta_tecmi = [_color_tecmi_light, _color_purple, _color_warning, _color_info, _color_primary]
_paleta_secundaria = [_color_tecmi_light, _color_primary, _color_purple]

## Carga del dataset desde URL pública

In [3]:
# definimos la url del archivo csv
_URL_ = 'https://raw.githubusercontent.com/RamRider89/analisis-estadistico-exploratorio/refs/heads/main/evidencia-1/healthcare_dataset.csv'
# leyendo el archivo csv mediante pandas
df = pd.read_csv(_URL)

## 🔍 Información de la estructura del DataSet

In [4]:
# definición de columnas por su tipo de estructura
_COLUMNAS_NUMERICAS_ = [_Age_, _Billing_Amount_, _Room_Number_]
_COLUMNAS_CATEGORICAS_ = [_Gender_, _Blood_Type_, _Medical_Condition_, _Insurance_Provider_, _Admission_Type_, _Medication_, _Test_Results_]
_COLUMNAS_TEXTO_ = [_Name_, _Doctor_, _Hospital_]
_COLUMNAS_FECHA_ = [_Date_Admission_, _Discharge_Date_]

In [5]:
print("\033[1m --- Dataset cargado correctamente --- \033[0m")
print("-"*80)
print(f"Dataset shape: {df.shape}")
print(f"Número de filas: {df.shape[0]}")
print(f"Número de columnas: {df.shape[1]}")

 --- Dataset cargado correctamente --- 
--------------------------------------------------------------------------------
Dataset shape: (55507, 15)
Número de filas: 55507
Número de columnas: 15


# 🩺 Diagnóstico y limpieza de registros mediante ETL

Se repite el proceso sistemático de ETL realizado en la Evidencia 1 para nuestro dataset, con el objetivo de limpieza y transformación de los datos.

### 🔡 Convertir las columnas de texto a minúsculas

Primeramente, se procede a convertir las variables tipo texto en minúsculas con el objetivo de asegurar la consistencia y uniformidad de los datos. De esta manera nos aseguramos de eliminar las variaciones en las cadenas de texto que podrían ser tratadas como registros únicos por algún algoritmo. Además, con esto aseguramos el encontrar y eliminar completamente los registros duplicados.

In [6]:
# en base a nuestras columnas de texto tales como _Name_, _Doctor_, _Hospital_
columnas_texto = df[_COLUMNAS_TEXTO_]

# recorremos las columnas para convertir sus valores a lowercase
for col in columnas_texto.columns:
    # asignación limpia de nuevos valores mediante loc
    df.loc[:, col] = df[col].str.lower()

## 🔴 Detección y eliminación de duplicados

Nuestro dataset **healthcare-dataset.csv** contiene registros duplicados que deben ser eliminados para evitar que sesguen el análisis.

In [7]:
print("\033[1m --- Limpieza de registros duplicados: --- \033[0m")
print("-" * 80)
print(f"Número de filas antes de la limpieza: {df.shape[0]}")
# contamos los registros duplicados
total_duplicados = df.duplicated().sum()
print(f"Registros \033[1m duplicados encontrados: {total_duplicados}\033[0m")
# eliminamos duplicados y nos aseguramos de que df_limpio sea una copia explícita
df_limpio = df.drop_duplicates().copy()
# resultados
print("-" * 80)
print(f"Número de \033[1m filas después de la limpieza: {df_limpio.shape[0]}\033[0m")
print(f"Número de columnas después de la limpieza: \033[1m {df_limpio.shape[1]}\033[0m")

 --- Limpieza de registros duplicados: --- 
--------------------------------------------------------------------------------
Número de filas antes de la limpieza: 55507
Registros  duplicados encontrados: 541
--------------------------------------------------------------------------------
Número de  filas después de la limpieza: 54966
Número de columnas después de la limpieza:  15


## 🔴 Eliminación de registros con Monto facturado negativo

🚨 Se ha detectado la presencia de valores negativos en la columna del monto facturado, la cuál es una anomalía significativa que definitivamente puede distorsionar nuestro análisis.

Los montos facturados negativos no tienen sentido en este contexto de servicios médicos y podrían deberse a errores en la entrada de datos.

Primeramente procederemos a calcular el número de registros negativos en el Dataset.

In [8]:
print("\033[1m --- Identificación de registros negativos --- \033[0m")
print("-" * 80)
# conteo
registros_negativos = df_limpio[df_limpio[_Billing_Amount_] < 0]
print(f"Registros con valores negativos: \033[1m {len(registros_negativos)} \033[0m")

 --- Identificación de registros negativos --- 
--------------------------------------------------------------------------------
Registros con valores negativos:  106 


🚨 Debido a que los valores negativos pueden ocacionar un sesgo en la media, la desviación estándar y el rango, además de afectar negativamente la precisión de las predicciones, he decidido optar por **eliminar los registros** que presenten este error.

Dado que tenemos 106 valores negativos en la columna Billing Amount, la mejor estrategia es eliminarlos. Este número es muy pequeño en comparación con el tamaño total de nuestro dataset, por lo que la eliminación no afectará significativamente la integridad del análisis.

In [9]:
print("\033[1m --- Eliminando registros negativos --- \033[0m")
print("-" * 80)
# eliminar las filas con valores negativos
df_limpio = df_limpio[df_limpio[_Billing_Amount_] >= 0]
# resultados
print(f"Número de \033[1m filas después de la limpieza: {df_limpio.shape[0]}\033[0m")
print(f"Número de columnas después de la limpieza: \033[1m {df_limpio.shape[1]}\033[0m")

 --- Eliminando registros negativos --- 
--------------------------------------------------------------------------------
Número de  filas después de la limpieza: 54841
Número de columnas después de la limpieza:  15


De esta manera podremos trabajar con un conjunto de datos más limpio y confiable, asegurando que los análisis y predicciones se basen únicamente en datos válidos y lógicos.

## 🔴 Detección y manejo de nulos

A continuación, identificaremos los valores nulos en el dataset y aplicaremos una estrategia de imputación para rellenar valores faltantes.

In [10]:
print("\033[1m --- Valores nulos por columna: --- \033[0m")
print("-" * 80)
# contamos todos los valores nulos de todas las columnas
nulos_por_columna = df_limpio.isnull().sum().reset_index()

# rajuste para mostrar los valores nulos como tabla
nulos_por_columna.columns=['Columna', 'Nulos']
print(nulos_por_columna.sort_values(by='Nulos', ascending=False))

 --- Valores nulos por columna: --- 
--------------------------------------------------------------------------------
               Columna  Nulos
1                  Age     22
2               Gender     16
3           Blood Type      6
4    Medical Condition      6
11      Admission Type      5
5    Date of Admission      0
0                 Name      0
6               Doctor      0
7             Hospital      0
9       Billing Amount      0
8   Insurance Provider      0
10         Room Number      0
12      Discharge Date      0
13          Medication      0
14        Test Results      0


A continuación, aplicaremos una imputación de valores utilizando la mediana para las columnas numéricas para evitar el sesgo de outliers.

Enseguida, aplicaremos una imputación en las columnas categóricas como 'Gender' y 'Medical Condition' con el valor más frecuente de estas variables.


### 🔢 Imputación de todas las columnas numericas

In [11]:
# imputando variables numéricas mediante el valor promedio
print("\033[1m --- Imputando variables numericas: --- \033[0m")
print("-" * 80)
# recorremos las columnas
for col in [_Age_, _Billing_Amount_]:
    # calculamos el valor promedio
    mediana = df_limpio[col].median()
    # rellenamos los valores nulos con la mediana, y utilizando .loc para una asignación segura
    df_limpio.loc[:, col] = df_limpio[col].fillna(mediana)
    # mensaje
    print(f"Inputando columna: \033[1m { col } \033[0m - con la mediana: \033[1m { str(mediana) } \033[0m")

 --- Imputando variables numericas: --- 
--------------------------------------------------------------------------------
Inputando columna:  Age  - con la mediana:  52.0 
Inputando columna:  Billing Amount  - con la mediana:  25590.7671438869 


### 🔡 Imputación de todas las columnas categóricas

In [12]:
# imputando variables categóricas mediante la moda
print("\033[1m --- Imputando variables categóricas: --- \033[0m")
print("-" * 80)
# recorremos las columnas
for col in [_Gender_, _Blood_Type_, _Medical_Condition_, _Admission_Type_ ]:
    # calculamos la moda de cada columna
    moda = df_limpio[col].mode()[0]
    # asignación segura mediante .loc
    df_limpio.loc[:, col] = df_limpio[col].fillna(moda)
    # mensaje a usuario
    print(f"Inputando columna: \033[1m { col } \033[0m - con el valor más frecuente: \033[1m { str(moda) } \033[0m ")

    # comprobacion
    #print(df_limpio.isnull().sum().loc[lambda x: x > 0].sort_values(ascending=False))

 --- Imputando variables categóricas: --- 
--------------------------------------------------------------------------------
Inputando columna:  Gender  - con el valor más frecuente:  Male  
Inputando columna:  Blood Type  - con el valor más frecuente:  A-  
Inputando columna:  Medical Condition  - con el valor más frecuente:  Arthritis  
Inputando columna:  Admission Type  - con el valor más frecuente:  Elective  


### ✅ Verificación de imputación



In [13]:
print("\033[1m --- Verificación de valores nulos por columna --- \033[0m")
print("-" * 80)
# contamos todos los valores nulos de todas las columnas
nulos_por_columna = df_limpio.isnull().sum().reset_index()
# rajuste para mostrar los valores nulos como tabla
nulos_por_columna.columns=['Columna', 'Nulos']
nulos_por_columna.sort_values(by='Nulos', ascending=False)

 --- Verificación de valores nulos por columna --- 
--------------------------------------------------------------------------------


,Columna,Nulos
0,Name,0
1,Age,0
2,Gender,0
3,Blood Type,0
4,Medical Condition,0
5,Date of Admission,0
6,Doctor,0
7,Hospital,0
8,Insurance Provider,0
9,Billing Amount,0


Se verifica que ya no contamos con ningún valor nulo.

### ✅ Verificación de valores únicos en columnas categóricas

In [14]:
# en base a nuestras columnas categoricas:_Gender_, _Blood_Type_, _Medical_Condition_, _Insurance_Provider_, _Admission_Type_, _Medication_, _Test_Results_
columnas_categoricas = df_limpio[_COLUMNAS_CATEGORICAS_]
# definimos el diccionario de valores unicos
_dict_unique_values_ = {
    _Gender_: [],
    _Blood_Type_: [],
    _Medical_Condition_: [],
    _Insurance_Provider_: [],
    _Admission_Type_: [],
    _Medication_: [],
    _Test_Results_: []
}

# recorremos las columnas categoricas para obtener sus valores
print("\033[1m --- Valores únicos por columna categórica: --- \033[0m")
print("-" * 80)
for col in columnas_categoricas.columns:
    # obtenemos los vals unicos
    unique_values = columnas_categoricas[col].unique()
    # asignamos los valores unicos al dict
    _dict_unique_values_[col] = columnas_categoricas[col].unique()

# para mostrar el resultado del diccionario en pantalla
# convertimos el diccionario a un conjunto de listas
data = [[key] + list(value) for key, value in _dict_unique_values_.items()]

# definimos los titulos de la tabla
headers = ["Columna"] + [f"Valor unico {i+1}" for i in range(len(max(_dict_unique_values_.values(), key=len)))]
print(tabulate(data, headers=headers, tablefmt="grid"))


 --- Valores únicos por columna categórica: --- 
--------------------------------------------------------------------------------
+--------------------+-----------------+-----------------+-----------------+------------------+-----------------+-----------------+-----------------+-----------------+
| Columna            | Valor unico 1   | Valor unico 2   | Valor unico 3   | Valor unico 4    | Valor unico 5   | Valor unico 6   | Valor unico 7   | Valor unico 8   |
+====================+=================+=================+=================+==================+=================+=================+=================+=================+
| Gender             | Male            | Female          |                 |                  |                 |                 |                 |                 |
+--------------------+-----------------+-----------------+-----------------+------------------+-----------------+-----------------+-----------------+-----------------+
| Blood Type         | B-     

De esta manera hemos comprobado los valores únicos de nuestras columnas categóricas, mismo que ya están limpios y sin valores nulos.

## ⚙️ Transformación y Normalización de Variables

A continuación, se procederá a transformar las variables para hacerlas útiles para el análisis y modelado.

### ✔ 1. Creación de una nueva variable: Duration of Stay

Se creará una nueva columna **Duration of Stay (Days)** la cual contendrá la duración en días de los pacientes dentro del hospital.

Para poder hacer este cálculo, las fechas de admisión y alta deben de ser convertidas a **datetime**.

#### Convirtiendo las fechas a datetime

In [15]:
# convirtiendo cada registro a datetime
# usamos .loc para realizar la conversión uno a uno
df_limpio.loc[:, _Date_Admission_] = pd.to_datetime(df_limpio.loc[:, _Date_Admission_], errors='coerce')
df_limpio.loc[:, _Discharge_Date_] = pd.to_datetime(df_limpio.loc[:, _Discharge_Date_], errors='coerce')
# convirtiendo el tipo de dato de la columna
df_limpio[_COLUMNAS_FECHA_] = df_limpio[_COLUMNAS_FECHA_].astype('datetime64[ns]')

#### Agregando nueva columna

In [16]:
# creando la nueva columna Duration of Stay (Days)
# esta nueva columna contendrá la duración en dias de los pacientes dentro del hospital
_Duration_Stay_ = 'Duration of Stay (Days)'

# calculando la duración de la estancia en días
df_limpio.loc[:, _Duration_Stay_] = (df_limpio.loc[:, _Discharge_Date_] - df_limpio.loc[:, _Date_Admission_]).dt.days

# verificando
print("\033[1m --- Verificando nueva columna --- \033[0m")
print("-" * 80)
_COLUMNAS_FECHA_ = [_Date_Admission_, _Discharge_Date_, _Duration_Stay_]
df_limpio[_COLUMNAS_FECHA_].dtypes

 --- Verificando nueva columna --- 
--------------------------------------------------------------------------------


,0
Date of Admission,datetime64[ns]
Discharge Date,datetime64[ns]
Duration of Stay (Days),int64


### ✔ 2. Normalización de la columna Billing Amount

La variable **Billing Amount** que corresponde al costo de atención médica, tiene un rango de valores muy amplio, lo que puede afectar negativamente a modelos de aprendizaje sensibles a la escala como K-Means.

La normalización (Min-Max) escalará los valores a un rango entre 0 y 1, lo que asegurará que todos los datos tengan el mismo peso en el análisis.

In [17]:
# se inicializa el escalador Min-Max
scaler = MinMaxScaler()
# definimos la nueva columna
_Billing_Amount_Normalized_ = 'Billing Amount (Normalized)'

# aplicamos la normalización a la columna
df_limpio.loc[:, _Billing_Amount_Normalized_] = scaler.fit_transform(df_limpio.loc[:,[_Billing_Amount_]])

### ✔ 3. Recodificación de Variables Categóricas

Los modelos de machine learning no pueden procesar variables categóricas como Gender y Medical Condition directamente, es por ello que realizaremos una recodificación automática a un formato numérico, lo cual es esencial para el modelado.

In [18]:
# inicializamos el codificador Label Encoding
le = LabelEncoder()

# nuevas columnas
_Gender_Encoded_ = 'Gender (Encoded)'
_Medical_Condition_Encoded_ = 'Medical Condition (Encoded)'

# aplicamos un label Encoding a la columna 'Gender'
df_limpio.loc[:, _Gender_Encoded_] = le.fit_transform(df_limpio.loc[:, _Gender_])

# aplicamos un label Encoding a la columna 'Medical Condition'
df_limpio.loc[:, _Medical_Condition_Encoded_] = le.fit_transform(df_limpio.loc[:, _Medical_Condition_])

# dict
_dict_unique_med_values_ = {
    _Medical_Condition_: df_limpio[_Medical_Condition_].unique(),
    _Medical_Condition_Encoded_: df_limpio[_Medical_Condition_Encoded_].unique()
}

# para mostrar el resultado del diccionario en pantalla
# convertimos el diccionario a un conjunto de listas
data = [[key] + list(value) for key, value in _dict_unique_med_values_.items()]

# definimos los titulos de la tabla
headers = ["Columna"] + [f"Valor unico {i+1}" for i in range(len(max(_dict_unique_med_values_.values(), key=len)))]

# imprimiendo
print("\033[1m --- Recodificación de Variables Categóricas --- \033[0m")
print("-" * 80)
print(tabulate(data, headers=headers, tablefmt="grid"))

 --- Recodificación de Variables Categóricas --- 
--------------------------------------------------------------------------------
+-----------------------------+-----------------+-----------------+-----------------+-----------------+-----------------+-----------------+
| Columna                     | Valor unico 1   | Valor unico 2   | Valor unico 3   | Valor unico 4   | Valor unico 5   | Valor unico 6   |
+=============================+=================+=================+=================+=================+=================+=================+
| Medical Condition           | Cancer          | Arthritis       | Obesity         | Diabetes        | Asthma          | Hypertension    |
+-----------------------------+-----------------+-----------------+-----------------+-----------------+-----------------+-----------------+
| Medical Condition (Encoded) | 2               | 0               | 5               | 3               | 1               | 4               |
+----------------------------

Al momento, se han realizado tres transformaciones:
1. **Creación de una nueva variable: Duration of Stay**
2. **Normalización de la columna Billing Amount**
3. **Recodificación de Variables Categóricas**

Además de la transformación inicial para **convertir todas las columas texto a lowercase.**

Con estas transformaciones, nuestro dataset está limpio y enriquecido.

# 🆕 Nuevas columnas para análisis multivariado o de series temporales

## Propuesta de nuevas columnas

Tome la decisión de añadir nuevas columnas que me ayuden en capturar la complejidad de la interacción del paciente con los gastos y las condiciones medicas. Esto será clave para mejorar el rendimiento de los modelos, especialmente después de que el modelo de regresión lineal simple fallará en la Evidencia 1 al utilizar dos variables clave.

Las nuevas columnas que se añadirán son:
1. Frecuencia de Visita: Me ayudará a identificar a los pacientes con riesgo crónico que utilizan los servicios del hospital de forma recurrente.
2. Medicación de Alto Riesgo: Esta variable me ayudará a identificar el uso de medicamentos que se asocian con enfermedades graves o de alto costo.
3. Combinación de Alto Riesgo: Esta columna me ayudará a identificar a los pacientes que reciben medicamentos de alto riesgo mientras tienen una condición médica crónica o grave.

## 📗 Conservación de columna Name
Se conserva la columna Name para poder generar y evaluar la nueva columna Frecuencia de Visita por paciente.

## 🔵 Nueva columna: Frecuencia de Visita

Esta variable ayudará a diferenciar entre pacientes con una sola admisión costosa y pacientes con múltiples admisiones de bajo costo pero alto costo acumulado.

In [19]:
# frecuencia de visita por el número de veces que acude el paciente
# nueva col
_Frequency_Visit_ = 'Frequency of Visit'
df_limpio[_Frequency_Visit_] = df_limpio.groupby(_Name_)[_Name_].transform('count')

## 🔵 Nueva columna: Medicación de Alto Riesgo

Recordemos que en nuestro dataset manejamos 5 medicamentos, los cuales son:
- Paracetamol
- Ibuprofen
- Aspirin
- Penicillin
- Lipitor

De acuerdo con Forhers (2025) *Lipitor es un medicamento a base de estatinas que se vende con receta y se usa para reducir el colesterol alto y ayudar a prevenir eventos cardiovasculares como ataques cardíacos o accidentes cerebrovasculares en personas con ciertos factores de riesgo.* **Por lo que será considerado nuestro medicamente de mayor riesgo.**


Seguido de la Penicillin por su fuerte uso como antibiótico.

Los tres medicamentos restantes no serán incluidos en la lista de alto riesgo o costo

In [20]:
# lista de medicamentos de alto riesgo/costo
high_risk_meds = ['Lipitor', 'Penicillin']

# nueva col
_High_Risk_Medication_ = 'High Risk Medication'

# columna binaria para identificar los riesgos
# 1 si toma medicamento de alto riesgo, 0 si no
df_limpio[_High_Risk_Medication_] = np.where(
    df_limpio[_Medication_].isin(high_risk_meds),
    1,
    0
)

Con esta nueva columna, podremos obtener un indicador clínico de alto riesgo, lo cual es mucho más fuerte que solo la Medical Condition general.

## 🔵 Nueva columna: Combinación de Alto Riesgo

Con la intención de mejorar significativamente la precisión de los modelos, he decido crear una variable que aplique una combinación de medicamentos de alto riesgo con enfermedades de alto riesgo.

Verificando en la columna: Medical Condition, nuestro dataset incluye a las enfermedades:
- Cancer
- Arthritis
- Obesity
- Diabetes
- Asthma
- Hypertension

De acuerdo con la Organización Mundial de la Salud (2023) *El cáncer es la principal causa de muerte en el mundo: en 2020 se atribuyeron a esta enfermedad casi 10 millones de defunciones, es decir, casi una de cada seis de las que se registran.*

Según la Organización Mundial de la Salud (2024) *El número de personas que viven con diabetes pasó de 200 millones en 1990 a 830 millones en 2022... Desde el año 2000, las tasas de mortalidad por diabetes han ido en aumento.*


De acuerdo con la Organización Mundial de la Salud (2023) *Se estima que en el mundo hay 1280 millones de adultos de 30 a 79 años con hipertensión... Entre otras complicaciones, la hipertensión puede producir daños cardiacos graves*


En relación con estos informes, se ha identificado que las condiciones que requieren manejo crónico, y por lo tanto se pueden catalogar como **alto riesgo** son:
- Cancer, Diabetes e Hypertension.

In [21]:
# lista de medicamentos de alto riesgo previamente definido
high_risk_meds = ['Lipitor', 'Penicillin']

# listado de condiciones medicas de alto riesgo
chronic_conditions = ['Cancer', 'Diabetes', 'Hypertension']

# nueva columna combina
_High_Risk_Combo_ = 'High Risk Combo'

# columna para la combinación de alto riesgo
# true si el paciente tiene una condición crónica y ademas usa un medicamento de alto riesgo
is_chronic = df_limpio[_Medical_Condition_].isin(chronic_conditions)
is_high_med = df_limpio[_Medication_].isin(high_risk_meds)

# nueva columna mediante condición combinada para ambas variables
df_limpio[_High_Risk_Combo_] = np.where(
    is_chronic & is_high_med,
    1,
    0
)

# verificando que las nuevas columnas se han creado
print("\033[1m --- Nuevas columnas de riesgo añadidas --- \033[0m")
print("-" * 80)
df_limpio[[_Name_,_Medication_, _Medical_Condition_, _Frequency_Visit_, _High_Risk_Medication_, _High_Risk_Combo_]].head()

 --- Nuevas columnas de riesgo añadidas --- 
--------------------------------------------------------------------------------


,Name,Medication,Medical Condition,Frequency of Visit,High Risk Medication,High Risk Combo
0,bobby jackson,Paracetamol,Cancer,1,0,0
1,leslie terry,Ibuprofen,Arthritis,1,0,0
2,danny smith,Aspirin,Obesity,1,0,0
3,andrew watts,Ibuprofen,Diabetes,1,0,0
4,adrienne bell,Penicillin,Cancer,2,1,1


Con estas tres nuevas variables de características, el dataset esta preparado para el análisis multivariable.

# ✅ Verificación de Dataset limpio

A continuación verificamos que la copia de nuestro dataset original, después de pasar por una serie de pasos de limpieza y transformación, este lista para trabajar.

### ✔ Estructura del Dataset limpio

In [22]:
print("\033[1m --- Dataset limpiado correctamente --- \033[0m")
print("-" * 80)
print(f"Número de filas: {df_limpio.shape[0]}")
print(f"Número de columnas: {df_limpio.shape[1]}")

 --- Dataset limpiado correctamente --- 
--------------------------------------------------------------------------------
Número de filas: 54841
Número de columnas: 22


In [23]:
df_limpio.info()

<class 'pandas.core.frame.DataFrame'>
Index: 54841 entries, 0 to 55499
Data columns (total 22 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   Name                         54841 non-null  object        
 1   Age                          54841 non-null  float64       
 2   Gender                       54841 non-null  object        
 3   Blood Type                   54841 non-null  object        
 4   Medical Condition            54841 non-null  object        
 5   Date of Admission            54841 non-null  datetime64[ns]
 6   Doctor                       54841 non-null  object        
 7   Hospital                     54841 non-null  object        
 8   Insurance Provider           54841 non-null  object        
 9   Billing Amount               54841 non-null  float64       
 10  Room Number                  54841 non-null  int64         
 11  Admission Type               54841 non-null  o

In [24]:
df_limpio.head()

,Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,...,Discharge Date,Medication,Test Results,Duration of Stay (Days),Billing Amount (Normalized),Gender (Encoded),Medical Condition (Encoded),Frequency of Visit,High Risk Medication,High Risk Combo
0,bobby jackson,30.0,Male,B-,Cancer,2024-01-31,matthew smith,sons and miller,Blue Cross,18856.281306,...,2024-02-02,Paracetamol,Normal,2,0.357256,1,2,1,0,0
1,leslie terry,62.0,Male,A-,Arthritis,2019-08-20,samantha davies,kim inc,Medicare,33643.327287,...,2019-08-26,Ibuprofen,Inconclusive,6,0.637552,1,0,1,0,0
2,danny smith,76.0,Female,A-,Obesity,2022-09-22,tiffany mitchell,cook plc,Aetna,27955.096079,...,2022-10-07,Aspirin,Normal,15,0.529729,0,5,1,0,0
3,andrew watts,28.0,Female,O+,Diabetes,2020-11-18,kevin wells,"hernandez rogers and vang,",Medicare,37909.782410,...,2020-12-18,Ibuprofen,Abnormal,30,0.718425,0,3,1,0,0
4,adrienne bell,43.0,Female,AB+,Cancer,2022-09-19,kathleen hanna,white-white,Aetna,14238.317814,...,2022-10-09,Penicillin,Abnormal,20,0.269720,0,2,2,1,1


### ✔ Verificación de Nulos en Dataset limpio

In [25]:
print(df_limpio.isnull().sum())

Name                           0
Age                            0
Gender                         0
Blood Type                     0
Medical Condition              0
Date of Admission              0
Doctor                         0
Hospital                       0
Insurance Provider             0
Billing Amount                 0
Room Number                    0
Admission Type                 0
Discharge Date                 0
Medication                     0
Test Results                   0
Duration of Stay (Days)        0
Billing Amount (Normalized)    0
Gender (Encoded)               0
Medical Condition (Encoded)    0
Frequency of Visit             0
High Risk Medication           0
High Risk Combo                0
dtype: int64


### 🗂 Resumen de la estructura del Dataset limpio

| Nombre de Columna			| Tipo de dato  | Rango / Valores únicos    												| Descripción
|---------------------------|---------------|---------------------------------------------------------------------------|---------------------------------------|
| Name 						| Text	 		| 54841 registros															| Edad del paciente. 					|
| Age 						| int64 		| 13 a 85 																	| Edad del paciente. 					|
| Gender 					| object 		| ['Male', 'Female'] 														| Género del paciente. 					|
| Blood Type 				| object 		| ['A+', 'A-', 'B+', 'B-', 'AB+', 'AB-', 'O+', 'O-'] 						| Tipo de sangre del paciente. 			|
| Medical Condition 		| object 		| [ Cancer - Arthritis - Obesity - Diabetes - Asthma - Hypertension ]		| Condición médica diagnosticada. 		|
| Date of Admission 		| object 		| 54841 registros	 														| Fecha en que el paciente fue admitido.|
| Doctor 					| object 		| 54841 registros 															| Nombre del médico a cargo. 			|
| Hospital 					| object 		| 54841 registros  															| Nombre del hospital de admisión. 		|
| Insurance Provider 		| object 		| 10 proveedores únicos 													| Compañía de seguros del paciente. 	|
| Billing Amount 			| float64 		| 54841 registros 	 														| Monto facturado por los servicios. 	|
| Room Number 				| int64 		| 54841 registros  	 														| Número de habitación asignada. 		|
| Admission Type 			| object 		| ['Emergency', 'Elective', 'Urgent'] 										| Tipo de admisión. 					|
| Discharge Date 			| object 		| 54841 registros  	 														| Fecha de alta del paciente. 			|
| Medication 				| object 		| 10 medicamentos únicos 													| Medicamento recetado. 				|
| Test Results 				| object 		| ['Normal', 'Abnormal', 'Inconclusive'] 									| Resultado de la prueba médica. 		|
| Duration of Stay Days		| int64 		| 54841 registros  	 														| Estancia en el hospital en dias. 		|
| Billing Amount Normalized | float64 		| 54841 registros  	 														| Monto facturado normalizado.	 		|
| Gender Encoded 			| int64 		| 54841 registros  	 														| Género del paciente (encoded). 		|
| Medical Condition Encoded	| int64 		| 54841 registros  	 														| Condición médica (encoded).			|
| Frequency of Visit   		| int64 		| 54841 registros  	 														| Frecuencia de visita por paciente.	|
| High Risk Medication		| Boolean		| 54841 registros  	 														| Paciente/medicamento de alto riesgo	|
| High Risk Combo 			| Boolean		| 54841 registros  	 														| Paciente/situación de alto riesgo		|

A partir del proceso de limpieza y transformación aplicado, el dataset es ahora mucho más robusto y está perfectamente preparado para el análisis multivariado.

## 1. Preparación y División de Datos

Un paso fundamental es estandarizar los datos continuos y dividirlos para evitar el sobreajuste.

In [31]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, r2_score, precision_score, recall_score, f1_score
import numpy as np
import warnings

# Ignorar advertencias de Scikit-learn
warnings.filterwarnings('ignore')

# --- 1. Preparación de Datos (Asumiendo que df_limpio y las variables codificadas existen) ---

# --- Definición de Variables (Asumiendo que existen en df_limpio) ---
# Variables Predictoras (X)
X = df_limpio[['Age', 'Duration of Stay (Days)', 'Gender (Encoded)',
               'Medical Condition (Encoded)', 'High Risk Medication',
               'Frequency of Visit', 'High Risk Combo']]

# Variables Objetivo (Y)
y_regresion = df_limpio['Billing Amount']         # Continua
y_clasificacion = df_limpio['High Risk Combo'] # Binaria (0 o 1)

# Estandarización de variables continuas (crucial para Logistic Regression y evitar sesgos)
scaler = StandardScaler()
# Estandarizamos solo las columnas que lo necesitan y que no son binarias (Gender, High Risk Combo, etc.)
X_cols_to_scale = ['Age', 'Duration of Stay (Days)', 'Frequency of Visit']
X_scaled = X.copy()
X_scaled[X_cols_to_scale] = scaler.fit_transform(X_scaled[X_cols_to_scale])

# División de datos para Regresión y Clasificación
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_scaled, y_regresion, test_size=0.3, random_state=42
)
X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X_scaled, y_clasificacion, test_size=0.3, random_state=42
)

## 2. Modelo de Regresión Lineal Múltiple (Predicción Continua)

Utilizamos LinearRegression para predecir el valor continuo del Billing Amount.

In [32]:
# --- 2. MODELO DE REGRESIÓN LINEAL MÚLTIPLE (Predicción Continua) ---

# Separación de datos para Regresión
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_scaled, y_regresion, test_size=0.3, random_state=42
)

# Entrenamiento del modelo
model_reg = LinearRegression()
model_reg.fit(X_train_reg, y_train_reg)

# Predicción
y_pred_reg = model_reg.predict(X_test_reg)

# Evaluación de Métricas de Regresión
mae = mean_absolute_error(y_test_reg, y_pred_reg)
r2 = r2_score(y_test_reg, y_pred_reg)
rmse = np.sqrt(mean_absolute_error(y_test_reg, y_pred_reg))

print("--- REGRESIÓN LINEAL MÚLTIPLE: PREDICCIÓN DE GASTOS ---")
print(f"R² (Varianza Explicada): {r2:.4f}")
print(f"Error Absoluto Medio (MAE): ${mae:,.2f}") # Error promedio en dólares
print(f"Raíz del Error Cuadrático Medio (RMSE): ${rmse:.2f}")

# Interpretación de Coeficientes
coeficientes = pd.DataFrame(model_reg.coef_, X.columns, columns=['Coeficiente'])
print("\nImpacto Cuantificado de las Variables:")
print(coeficientes.sort_values(by='Coeficiente', ascending=False))

--- REGRESIÓN LINEAL MÚLTIPLE: PREDICCIÓN DE GASTOS ---
R² (Varianza Explicada): -0.0000
Error Absoluto Medio (MAE): $12,288.66
Raíz del Error Cuadrático Medio (RMSE): $110.85

Impacto Cuantificado de las Variables:
                             Coeficiente
Gender (Encoded)              103.903836
Medical Condition (Encoded)    58.899199
Frequency of Visit             -2.254900
Age                           -36.316061
Duration of Stay (Days)       -46.768970
High Risk Combo              -144.313396
High Risk Medication         -227.780497


# Interpretación

El $\text{R}^2$ (porcentaje de la varianza explicada) indica qué tan bien se ajusta el modelo. Si es bajo (cercano a 0), el modelo es pobre. El MAE cuantifica el error promedio de la predicción en dólares, lo que es la métrica más útil para el área de finanzas. Los coeficientes estimados muestran cuánto cambia el Billing Amount por cada unidad de cambio en el predictor.

# 3. Modelo de Regresión Logística Binaria (Predicción Categórica)

In [33]:
# --- 3. MODELO DE REGRESIÓN LOGÍSTICA BINARIA (Predicción de Clase) ---

# Separación de datos para Clasificación
X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X_scaled, y_clasificacion, test_size=0.3, random_state=42
)

# Entrenamiento del modelo (usamos C para regularización L2 por defecto)
model_log = LogisticRegression(C=1.0, solver='liblinear', random_state=42)
model_log.fit(X_train_cls, y_train_cls)

# Predicción de clase (0 o 1)
y_pred_cls = model_log.predict(X_test_cls)

# Evaluación de Métricas de Clasificación
prec = precision_score(y_test_cls, y_pred_cls, zero_division=0)
rec = recall_score(y_test_cls, y_pred_cls, zero_division=0)
f1 = f1_score(y_test_cls, y_pred_cls, zero_division=0)

print("\n--- REGRESIÓN LOGÍSTICA BINARIA: PREDICCIÓN DE RIESGO ALTO (1) ---")
print(f"Precision (Evita Falsos Positivos): {prec:.4f}")
print(f"Recall (Sensibilidad/Detecta Riesgo Real): {rec:.4f}")
print(f"F1 Score (Equilibrio): {f1:.4f}")

# Interpretación de Coeficientes (Odds Ratios para impacto de la probabilidad)
# Exponenciar los coeficientes para obtener Odds Ratios
odds_ratios = pd.Series(np.exp(model_log.coef_[0]), index=X.columns)
print("\nOdds Ratios (Impacto en la Probabilidad de Alto Riesgo):")
print(odds_ratios.sort_values(ascending=False))

# --- NOTA SOBRE MULTINOMIAL Y ORDINAL ---
# Si tuviéramos una variable objetivo con 3+ clases sin orden (ej. Test Results: Normal, Abnormal, Inconclusive)
# se usaría: LogisticRegression(multi_class='multinomial', solver='lbfgs')
# Si tuviéramos una variable objetivo con 3+ clases con orden (ej. Satisfacción: Baja, Media, Alta)
# se usaría: modelos especializados como scikit-learn's LogisticRegressionCV (o bibliotecas específicas para ordinal)


--- REGRESIÓN LOGÍSTICA BINARIA: PREDICCIÓN DE RIESGO ALTO (1) ---
Precision (Evita Falsos Positivos): 1.0000
Recall (Sensibilidad/Detecta Riesgo Real): 1.0000
F1 Score (Equilibrio): 1.0000

Odds Ratios (Impacto en la Probabilidad de Alto Riesgo):
High Risk Combo                438961.657965
High Risk Medication                3.808553
Frequency of Visit                  1.001442
Age                                 0.996137
Duration of Stay (Days)             0.992265
Medical Condition (Encoded)         0.922799
Gender (Encoded)                    0.678284
dtype: float64


## Interpretación:
Para la clasificación, las métricas son vitales para la gestión de riesgos:

- Precision: Mide la confiabilidad de la predicción de "Alto Riesgo". Si es alta, minimizamos el riesgo de acusar a un paciente de ser de alto riesgo cuando no lo es (Falso Positivo).
- Recall: Mide la capacidad del modelo para detectar todos los casos de alto riesgo reales. Si es alta, minimizamos el riesgo de que un caso de alto riesgo se filtre sin intervención (Falso Negativo).
- Odds Ratios: Un valor superior a 1 en el Odds Ratio indica que esa variable aumenta la probabilidad de que la variable objetivo sea 1 (Alto Riesgo). Un valor inferior a 1 disminuye la probabilidad.

# Notas sobre Regresión Logística Multinomial y Ordinal

- Multinomial: Se utiliza cuando el objetivo tiene más de dos clases sin orden (ej., Test Results: Normal, Abnormal, Inconclusive). En Scikit-learn, esto se resuelve estableciendo multi_class='multinomial' y solver='lbfgs'.
- Ordinal: Se utiliza para clases con un orden (ej., Nivel de Satisfacción: Bajo, Medio, Alto). Scikit-learn no tiene un modelo nativo para esto, por lo que se suelen usar modelos especializados o se descompone el problema en múltiples clasificaciones binarias.

# Aplicación en Python de la regresión polinomial y sobreajuste

In [34]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso # Importar Ridge y Lasso
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score, precision_score, recall_score, f1_score
import numpy as np
import warnings

# Ignorar advertencias de Scikit-learn que no afectan la ejecución
warnings.filterwarnings('ignore')

# --- 1. Preparación de Datos (Asumiendo que df_limpio y las variables codificadas existen) ---

# Variables Predictoras (X) - Se usan variables continuas y categóricas codificadas
X = df_limpio[['Age', 'Duration of Stay (Days)', 'Gender (Encoded)',
               'Medical Condition (Encoded)', 'High Risk Medication',
               'Frequency of Visit', 'High Risk Combo']]

# Variables Objetivo (Y)
y_regresion = df_limpio['Billing Amount'] # Variable continua para Regresión
y_clasificacion = df_limpio['High Risk Combo'] # Variable binaria para Regresión Logística (0 o 1)

# Estandarización de Variables Continuas (Crucial para muchos modelos, especialmente regularización)
scaler = StandardScaler()
# Estandarizamos solo las columnas que lo necesitan y que no son binarias (Gender, High Risk Combo, etc.)
X_cols_to_scale = ['Age', 'Duration of Stay (Days)', 'Frequency of Visit']
X_scaled = X.copy()
X_scaled[X_cols_to_scale] = scaler.fit_transform(X_scaled[X_cols_to_scale])


# --- 2. MODELO DE REGRESIÓN LINEAL MÚLTIPLE (Predicción Continua) ---

# Separación de datos para Regresión
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_scaled, y_regresion, test_size=0.3, random_state=42
)

# Entrenamiento del modelo
model_reg = LinearRegression()
model_reg.fit(X_train_reg, y_train_reg)

# Predicción
y_pred_reg = model_reg.predict(X_test_reg)

# Evaluación de Métricas de Regresión
mae = mean_absolute_error(y_test_reg, y_pred_reg)
r2 = r2_score(y_test_reg, y_pred_reg)
rmse = np.sqrt(mean_absolute_error(y_test_reg, y_pred_reg))

print("--- REGRESIÓN LINEAL MÚLTIPLE: PREDICCIÓN DE GASTOS ---")
print(f"R² (Varianza Explicada): {r2:.4f}")
print(f"Error Absoluto Medio (MAE): ${mae:,.2f}") # Error en las unidades originales ($)
print(f"Raíz del Error Cuadrático Medio (RMSE): ${rmse:.2f}")

# Interpretación de Coeficientes (Impacto de cada variable)
coeficientes = pd.DataFrame(model_reg.coef_, X.columns, columns=['Coeficiente'])
print("\nImpacto Cuantificado de las Variables (Regresión Lineal Sin Regularizar):")
print(coeficientes.sort_values(by='Coeficiente', ascending=False))


# --- 2.1 REGULARIZACIÓN (Lasso L1 y Ridge L2) ---

# Aplicar Regularización Ridge (L2)
# Alpha (lambda) es el parámetro de penalización. Se puede ajustar.
ridge_model = Ridge(alpha=100.0)
ridge_model.fit(X_train_reg, y_train_reg)
ridge_pred = ridge_model.predict(X_test_reg)
ridge_r2 = r2_score(y_test_reg, ridge_pred)

print("\n--- REGULARIZACIÓN RIDGE (L2) ---")
print(f"R² (Ridge): {ridge_r2:.4f}")
ridge_coefs = pd.Series(ridge_model.coef_, index=X.columns).sort_values(ascending=False)
print("Coeficientes (Reducidos, no cero):")
print(ridge_coefs)


# Aplicar Regularización Lasso (L1)
# Un alpha más alto genera más coeficientes cero (selección de características).
lasso_model = Lasso(alpha=5000.0)
lasso_model.fit(X_train_reg, y_train_reg)
lasso_pred = lasso_model.predict(X_test_reg)
lasso_r2 = r2_score(y_test_reg, lasso_pred)

print("\n--- REGULARIZACIÓN LASSO (L1) ---")
print(f"R² (Lasso): {lasso_r2:.4f}")
lasso_coefs = pd.Series(lasso_model.coef_, index=X.columns).sort_values(ascending=False)
print("Coeficientes (Selección de Características, algunos serán cero):")
print(lasso_coefs)


# --- 3. MODELO DE REGRESIÓN LOGÍSTICA BINARIA (Predicción de Clase) ---

# Separación de datos para Clasificación
X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X_scaled, y_clasificacion, test_size=0.3, random_state=42
)

# Entrenamiento del modelo (usando C=0.1 para regularización simple)
model_log = LogisticRegression(C=0.1, solver='liblinear', random_state=42)
model_log.fit(X_train_cls, y_train_cls)

# Predicción de clase (0 o 1)
y_pred_cls = model_log.predict(X_test_cls)

# Evaluación de Métricas de Clasificación
acc = accuracy_score(y_test_cls, y_pred_cls)
prec = precision_score(y_test_cls, y_pred_cls, zero_division=0)
rec = recall_score(y_test_cls, y_pred_cls, zero_division=0)
f1 = f1_score(y_test_cls, y_pred_cls, zero_division=0)


print("\n\n--- REGRESIÓN LOGÍSTICA BINARIA: PREDICCIÓN DE RIESGO ALTO (1) ---")
print(f"Accuracy Global: {acc:.4f}")
print(f"Precision (Evita Falsos Positivos): {prec:.4f}") # De los que predijimos como 'Alto Riesgo', ¿cuántos acertamos?
print(f"Recall (Sensibilidad): {rec:.4f}") # De todos los casos de 'Alto Riesgo', ¿cuántos detectamos?
print(f"F1 Score: {f1:.4f}")

# Interpretación de Coeficientes (Odds Ratios para impacto de la probabilidad)
# Exponenciar los coeficientes para obtener Odds Ratios
odds_ratios = pd.Series(np.exp(model_log.coef_[0]), index=X.columns)
print("\nOdds Ratios (Impacto en la Probabilidad de Alto Riesgo):")
print(odds_ratios.sort_values(ascending=False))

# --- NOTA SOBRE MULTINOMIAL Y ORDINAL ---
# Si tuviéramos una variable objetivo con 3+ clases sin orden (ej. Test Results: Normal, Abnormal, Inconclusive)
# se usaría: LogisticRegression(multi_class='multinomial', solver='lbfgs')
# Si tuviéramos una variable objetivo con 3+ clases con orden (ej. Satisfacción: Baja, Media, Alta)
# se usaría: modelos especializados como scikit-learn's LogisticRegressionCV (o bibliotecas específicas para ordinal)


--- REGRESIÓN LINEAL MÚLTIPLE: PREDICCIÓN DE GASTOS ---
R² (Varianza Explicada): -0.0000
Error Absoluto Medio (MAE): $12,288.66
Raíz del Error Cuadrático Medio (RMSE): $110.85

Impacto Cuantificado de las Variables (Regresión Lineal Sin Regularizar):
                             Coeficiente
Gender (Encoded)              103.903836
Medical Condition (Encoded)    58.899199
Frequency of Visit             -2.254900
Age                           -36.316061
Duration of Stay (Days)       -46.768970
High Risk Combo              -144.313396
High Risk Medication         -227.780497

--- REGULARIZACIÓN RIDGE (L2) ---
R² (Ridge): -0.0000
Coeficientes (Reducidos, no cero):
Gender (Encoded)               102.813808
Medical Condition (Encoded)     58.817680
Frequency of Visit              -2.243940
Age                            -36.220918
Duration of Stay (Days)        -46.646463
High Risk Combo               -143.422711
High Risk Medication          -225.765319
dtype: float64

--- REGULARIZACIÓN LA

# Implementación de Regresión Polinomial y Comparación de Regularización


In [35]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures # Importar PolynomialFeatures
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score, precision_score, recall_score, f1_score
import numpy as np
import warnings

# Ignorar advertencias de Scikit-learn que no afectan la ejecución
warnings.filterwarnings('ignore')

# --- 1. Preparación de Datos (Asumiendo que df_limpio y las variables codificadas existen) ---

# Variables Predictoras (X) - Se usan variables continuas y categóricas codificadas
X = df_limpio[['Age', 'Duration of Stay (Days)', 'Gender (Encoded)',
               'Medical Condition (Encoded)', 'High Risk Medication',
               'Frequency of Visit', 'High Risk Combo']]

# Variables Objetivo (Y)
y_regresion = df_limpio['Billing Amount'] # Variable continua para Regresión
y_clasificacion = df_limpio['High Risk Combo'] # Variable binaria para Regresión Logística (0 o 1)

# Estandarización de Variables Continuas (Crucial para muchos modelos, especialmente regularización)
scaler = StandardScaler()
# Estandarizamos solo las columnas que lo necesitan y que no son binarias (Gender, High Risk Combo, etc.)
X_cols_to_scale = ['Age', 'Duration of Stay (Days)', 'Frequency of Visit']
X_scaled = X.copy()
X_scaled[X_cols_to_scale] = scaler.fit_transform(X_scaled[X_cols_to_scale])


# --- 2. MODELO DE REGRESIÓN LINEAL MÚLTIPLE (Predicción Continua) ---

# Separación de datos para Regresión
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_scaled, y_regresion, test_size=0.3, random_state=42
)

# Entrenamiento del modelo
model_reg = LinearRegression()
model_reg.fit(X_train_reg, y_train_reg)

# Evaluación de Métricas de Regresión
mae = mean_absolute_error(y_test_reg, y_pred_reg)
r2 = r2_score(y_test_reg, y_pred_reg)
rmse = np.sqrt(mean_absolute_error(y_test_reg, y_pred_reg))


print("--- REGRESIÓN LINEAL MÚLTIPLE: PREDICCIÓN DE GASTOS ---")
print(f"R² (Varianza Explicada): {r2:.4f}")
print(f"Error Absoluto Medio (MAE): ${mae:,.2f}") # Error en las unidades originales ($)


# --- 2.1 REGRESIÓN POLINOMIAL (Grado 2 para modelar no linealidad) ---

# Creación de características polinomiales de grado 2 (X^2) y términos de interacción
poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train_reg)
X_test_poly = poly.transform(X_test_reg)

# Entrenar el modelo lineal sobre las características polinomiales
model_poly = LinearRegression()
model_poly.fit(X_train_poly, y_train_reg)
y_pred_poly = model_poly.predict(X_test_poly)
r2_poly = r2_score(y_test_reg, y_pred_poly)

print("\n--- REGRESIÓN POLINOMIAL (Grado 2) ---")
print(f"R² (Polinomial Grado 2): {r2_poly:.4f}")
# Nota: Un aumento significativo aquí indicaría una relación curva.


# --- 2.2 REGULARIZACIÓN (Lasso L1 y Ridge L2) y COMPARACIÓN DE COEFICIENTES ---

# Aplicar Regularización Ridge (L2) sobre el modelo polinomial para reducir la complejidad
ridge_model = Ridge(alpha=100.0)
ridge_model.fit(X_train_poly, y_train_reg)
ridge_r2 = r2_score(y_test_reg, ridge_model.predict(X_test_poly))

print("\n--- REGULARIZACIÓN RIDGE (L2) sobre Polinomial ---")
print(f"R² (Ridge): {ridge_r2:.4f}")
print("Ridge reduce coeficientes sin llevarlos a cero.")


# Aplicar Regularización Lasso (L1) sobre el modelo polinomial para SELECCIÓN DE CARACTERÍSTICAS
lasso_model = Lasso(alpha=5000.0) # Alpha alto para forzar la eliminación de variables
lasso_model.fit(X_train_poly, y_train_reg)
lasso_r2 = r2_score(y_test_reg, lasso_model.predict(X_test_poly))

print("\n--- REGULARIZACIÓN LASSO (L1) sobre Polinomial ---")
print(f"R² (Lasso): {lasso_r2:.4f}")
print("Lasso fuerza coeficientes a cero, eliminando variables irrelevantes.")

# Comparación del número de coeficientes (ejemplo de cómo se evalúa la simplificación)
num_coefs_lasso = np.sum(lasso_model.coef_ != 0)
print(f"Total de características usadas por Lasso (coef != 0): {num_coefs_lasso} (Demuestra simplificación)")


# --- 3. MODELO DE REGRESIÓN LOGÍSTICA BINARIA (Predicción de Clase) ---

# Separación de datos para Clasificación
X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X_scaled, y_clasificacion, test_size=0.3, random_state=42
)

# Entrenamiento del modelo (usando C=0.1 para regularización simple)
model_log = LogisticRegression(C=0.1, solver='liblinear', random_state=42)
model_log.fit(X_train_cls, y_train_cls)

# Predicción de clase (0 o 1)
y_pred_cls = model_log.predict(X_test_cls)

# Evaluación de Métricas de Clasificación
acc = accuracy_score(y_test_cls, y_pred_cls)
prec = precision_score(y_test_cls, y_pred_cls, zero_division=0)
rec = recall_score(y_test_cls, y_pred_cls, zero_division=0)
f1 = f1_score(y_test_cls, y_pred_cls, zero_division=0)


print("\n\n--- REGRESIÓN LOGÍSTICA BINARIA: PREDICCIÓN DE RIESGO ALTO (1) ---")
print(f"Accuracy Global: {acc:.4f}")
print(f"Precision (Evita Falsos Positivos): {prec:.4f}") # De los que predijimos como 'Alto Riesgo', ¿cuántos acertamos?
print(f"Recall (Sensibilidad): {rec:.4f}") # De todos los casos de 'Alto Riesgo', ¿cuántos detectamos?
print(f"F1 Score: {f1:.4f}")

# Interpretación de Coeficientes (Odds Ratios para impacto de la probabilidad)
# Exponenciar los coeficientes para obtener Odds Ratios
odds_ratios = pd.Series(np.exp(model_log.coef_[0]), index=X.columns)
print("\nOdds Ratios (Impacto en la Probabilidad de Alto Riesgo):")
print(odds_ratios.sort_values(ascending=False))


--- REGRESIÓN LINEAL MÚLTIPLE: PREDICCIÓN DE GASTOS ---
R² (Varianza Explicada): -0.0000
Error Absoluto Medio (MAE): $12,288.66

--- REGRESIÓN POLINOMIAL (Grado 2) ---
R² (Polinomial Grado 2): -0.0012

--- REGULARIZACIÓN RIDGE (L2) sobre Polinomial ---
R² (Ridge): -0.0011
Ridge reduce coeficientes sin llevarlos a cero.

--- REGULARIZACIÓN LASSO (L1) sobre Polinomial ---
R² (Lasso): -0.0000
Lasso fuerza coeficientes a cero, eliminando variables irrelevantes.
Total de características usadas por Lasso (coef != 0): 0 (Demuestra simplificación)


--- REGRESIÓN LOGÍSTICA BINARIA: PREDICCIÓN DE RIESGO ALTO (1) ---
Accuracy Global: 1.0000
Precision (Evita Falsos Positivos): 1.0000
Recall (Sensibilidad): 1.0000
F1 Score: 1.0000

Odds Ratios (Impacto en la Probabilidad de Alto Riesgo):
High Risk Combo                9233.247342
High Risk Medication              3.417577
Frequency of Visit                1.001219
Age                               0.995782
Duration of Stay (Days)           0.99242